# MODELO ESTRELLA

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month, dayofmonth, hour
spark = SparkSession.builder.appName("refined_modelo_estrella").getOrCreate()

## LEEMOS EL PARQUET LIMPIO DESDE TRUSTED

In [3]:
df_trusted = spark.read.parquet("gs://retail-transactions-final/trusted/retail_trusted.parquet")

df_trusted.printSchema()
df_trusted.show(5)

root
 |-- Transaction_ID: long (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Total_Items: integer (nullable = true)
 |-- Total_Cost: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Discount_Applied: string (nullable = true)
 |-- Customer_Category: string (nullable = true)
 |-- Season: string (nullable = true)
 |-- Promotion: string (nullable = true)



+--------------+-------------------+-----------------+--------------------+-----------+----------+--------------+-------------+----------------+----------------+-----------------+------+--------------------+
|Transaction_ID|               Date|    Customer_Name|             Product|Total_Items|Total_Cost|Payment_Method|         City|      Store_Type|Discount_Applied|Customer_Category|Season|           Promotion|
+--------------+-------------------+-----------------+--------------------+-----------+----------+--------------+-------------+----------------+----------------+-----------------+------+--------------------+
|    1000000000|2022-01-21 06:27:29|     Stacey Price|['Ketchup', 'Shav...|          3|     71.65|Mobile Payment|  Los Angeles|  Warehouse Club|            True|        Homemaker|Winter|                None|
|    1000000001|2023-03-01 13:01:21| Michelle Carlson|['Ice Cream', 'Mi...|          2|     25.93|          Cash|San Francisco| Specialty Store|            True|     Pr

## 1. DIMENSION TIEMPO

In [6]:
dim_tiempo = df_trusted.select("Date", "Season") \
    .withColumn("anio", year("Date")) \
    .withColumn("mes", month("Date")) \
    .withColumn("dia", dayofmonth("Date")) \
    .withColumn("hora", hour("Date")) \
    .dropDuplicates(["Date"])

In [7]:
# lO GUARDAMOS
dim_tiempo.write.mode("overwrite").parquet("gs://retail-transactions-final/refined/dimensiones/dim_tiempo")

## 2. DIMENSIÓN CLIENTE

In [8]:
dim_cliente = df_trusted.select("Customer_Name", "Customer_Category").dropDuplicates()
dim_cliente.write.mode("overwrite").parquet("gs://retail-transactions-final/refined/dimensiones/dim_cliente")

## 3. DIMENSIÓN TIENDA

In [ ]:
dim_tienda = df_trusted.select("City", "Store_Type").dropDuplicates()
dim_tienda.write.mode("overwrite").parquet("gs://retail-transactions-final/refined/dimensiones/dim_tienda")

## 4. TABLA DE HECHOS

In [4]:
fact_transacciones = df_trusted.select(
    "Transaction_ID",
    "Date",
    "Customer_Name",
    "Total_Items",
    "Total_Cost",
    "Payment_Method",
    "City",
    "Store_Type",
    "Promotion"
)

fact_transacciones.write.mode("overwrite").parquet("gs://retail-transactions-final/refined/hechos/fact_transacciones")